In [1]:
!pip install scikit-surprise

   ---------------------------------------- 0.0/1.3 MB ? eta -:--:--
   ---------------------------------------- 0.0/1.3 MB ? eta -:--:--
   ---------------------------------------- 1.3/1.3 MB 8.6 MB/s  0:00:00


In [2]:
# ============================================================================
# RATING PREDICTION: Bias MF vs SVD++
# ============================================================================
# - Explicit feedback, MSE loss, KHÔNG negative sampling
# - Preprocessing: LOO split (1 test rating/user), giữ rating values
# - Eval: RMSE, MAE, R² trên test
# - Đo: train_time, eval_time
# ============================================================================

import time, math
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from pathlib import Path
from datetime import datetime
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score


# ============================================================
# 1. CONFIG
# ============================================================
class Config:
    DATA_DIR       = Path("../data")
    FINAL_DATA_DIR = DATA_DIR / "final_data"
    ARTIFACT_DIR   = DATA_DIR / "artifacts"
    CF_DIR         = ARTIFACT_DIR / "cf"
    RATINGS_PATH   = FINAL_DATA_DIR / "ratings_filtered_by_movie.csv"

    USER_COL      = "userId"
    MOVIE_COL     = "movieId"
    RATING_COL    = "rating"
    TIMESTAMP_COL = "timestamp"

    MAX_ROWS_TO_PROCESS = 1_000_000
    MIN_INTERACTIONS    = 20

    EMBEDDING_DIM = 64
    BATCH_SIZE    = 4096
    EPOCHS        = 30
    LEARNING_RATE = 1e-3
    WEIGHT_DECAY  = 1e-5
    PATIENCE      = 3

    DEVICE        = "cuda" if torch.cuda.is_available() else "cpu"
    RANDOM_SEED   = 42
    # Tiền tố Versioning bằng Timestamp
    RUN_TIMESTAMP = f"{datetime.now().strftime('%Y%m%d_%H%M%S')}_{1_000_000}"

Config.CF_DIR.mkdir(parents=True, exist_ok=True)
RUN_DIR = Config.CF_DIR / Config.RUN_TIMESTAMP
RUN_DIR.mkdir(exist_ok=True)
DEVICE = torch.device(Config.DEVICE)

print(f"[Config] Device={Config.DEVICE} | Run={Config.RUN_TIMESTAMP}")


def set_seed(seed=42):
    import random
    random.seed(seed); np.random.seed(seed)
    torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


# ============================================================
# 2. PREPROCESSING (giữ nguyên logic hiện tại)
# ============================================================
def preprocess(csv_path, max_rows=1_000_000, min_interactions=20):
    print(f"[Prep] Reading {max_rows:,} newest...")
    recent = pd.DataFrame()
    for chunk in pd.read_csv(
        csv_path,
        usecols=[Config.USER_COL, Config.MOVIE_COL, Config.RATING_COL, Config.TIMESTAMP_COL],
        chunksize=200_000,
    ):
        recent = pd.concat([recent, chunk], ignore_index=True)
        recent = recent.nlargest(max_rows, Config.TIMESTAMP_COL)

    recent = recent.dropna()
    counts = recent[Config.USER_COL].value_counts()
    users_keep = counts[counts >= min_interactions].index
    df = recent[recent[Config.USER_COL].isin(users_keep)].copy()

    df = df.sort_values(
        [Config.USER_COL, Config.TIMESTAMP_COL, Config.MOVIE_COL], kind="stable"
    ).reset_index(drop=True)

    user_map  = {str(u): i for i, u in enumerate(df[Config.USER_COL].unique())}
    movie_map = {str(m): i for i, m in enumerate(df[Config.MOVIE_COL].unique())}
    df["user_idx"]  = df[Config.USER_COL].astype(str).map(user_map)
    df["movie_idx"] = df[Config.MOVIE_COL].astype(str).map(movie_map)

    train_parts, val_parts, test_parts = [], [], []
    for uid, grp in df.groupby("user_idx", sort=False):
        train_parts.append(grp.iloc[:-2])
        val_parts.append(grp.iloc[-2:-1])
        test_parts.append(grp.iloc[-1:])

    train_df = pd.concat(train_parts, ignore_index=True)
    val_df   = pd.concat(val_parts,   ignore_index=True)
    test_df  = pd.concat(test_parts,  ignore_index=True)

    print(f"[Prep] Train={len(train_df):,} | Val={len(val_df):,} | Test={len(test_df):,}")
    print(f"[Prep] Users={len(user_map):,} | Items={len(movie_map):,} | "
          f"Rating range=[{train_df[Config.RATING_COL].min()}, {train_df[Config.RATING_COL].max()}]")
    return train_df, val_df, test_df, user_map, movie_map


# ============================================================
# 3. BIAS MF
# ============================================================
class BiasMF(nn.Module):
    """r_ui = mu + b_u + b_i + p_u^T q_i"""
    name = "Bias MF"

    def __init__(self, num_users, num_movies, dim=64, init_std=0.01):
        super().__init__()
        self.user_emb    = nn.Embedding(num_users, dim)
        self.item_emb    = nn.Embedding(num_movies, dim)
        self.user_bias   = nn.Embedding(num_users, 1)
        self.item_bias   = nn.Embedding(num_movies, 1)
        self.global_bias = nn.Parameter(torch.zeros(1))

        nn.init.normal_(self.user_emb.weight, std=init_std)
        nn.init.normal_(self.item_emb.weight, std=init_std)
        nn.init.zeros_(self.user_bias.weight)
        nn.init.zeros_(self.item_bias.weight)

    def forward(self, u, i):
        return ((self.user_emb(u) * self.item_emb(i)).sum(-1)
                + self.user_bias(u).squeeze(-1)
                + self.item_bias(i).squeeze(-1)
                + self.global_bias)


# ============================================================
# 4. SVD++
# ============================================================
class SVDpp(nn.Module):
    """r_ui = mu + b_u + b_i + q_i^T (p_u + |N(u)|^{-1/2} * sum_{j in N(u)} y_j)"""
    name = "SVD++"

    def __init__(self, num_users, num_movies, dim=64, init_std=0.01):
        super().__init__()
        self.user_emb          = nn.Embedding(num_users, dim)
        self.item_emb          = nn.Embedding(num_movies, dim)
        self.implicit_item_emb = nn.Embedding(num_movies, dim)   # y_j
        self.user_bias         = nn.Embedding(num_users, 1)
        self.item_bias         = nn.Embedding(num_movies, 1)
        self.global_bias       = nn.Parameter(torch.zeros(1))

        nn.init.normal_(self.user_emb.weight, std=init_std)
        nn.init.normal_(self.item_emb.weight, std=init_std)
        nn.init.normal_(self.implicit_item_emb.weight, std=init_std)
        nn.init.zeros_(self.user_bias.weight)
        nn.init.zeros_(self.item_bias.weight)

    def implicit_sum(self, unique_users_np, user_items_dict):
        """Compute |N(u)|^{-1/2} * sum(y_j) for each unique user via scatter_add."""
        all_user_pos = []     # position in unique_users tensor
        all_items    = []
        for pos, uid in enumerate(unique_users_np):
            items = user_items_dict.get(int(uid), [])
            all_user_pos.extend([pos] * len(items))
            all_items.extend(items)

        if not all_items:
            return torch.zeros(len(unique_users_np), self.user_emb.embedding_dim,
                               device=DEVICE)

        pos_t   = torch.tensor(all_user_pos, dtype=torch.long, device=DEVICE)
        items_t = torch.tensor(all_items,    dtype=torch.long, device=DEVICE)

        y_j = self.implicit_item_emb(items_t)                       # (total_N, dim)
        dim = y_j.size(-1)

        implicit_sums = torch.zeros(len(unique_users_np), dim, device=DEVICE)
        implicit_sums.scatter_add_(0, pos_t.unsqueeze(-1).expand(-1, dim), y_j)

        counts = torch.zeros(len(unique_users_np), device=DEVICE)
        counts.scatter_add_(0, pos_t, torch.ones_like(pos_t, dtype=torch.float))
        norm = 1.0 / torch.sqrt(counts + 1e-8)

        return implicit_sums * norm.unsqueeze(-1)

    def forward(self, u, i, implicit_per_batch):
        """implicit_per_batch: (B, dim) — implicit adjustment for each (u,i) instance."""
        p_u = self.user_emb(u)
        q_i = self.item_emb(i)
        user_factor = p_u + implicit_per_batch
        return ((user_factor * q_i).sum(-1)
                + self.user_bias(u).squeeze(-1)
                + self.item_bias(i).squeeze(-1)
                + self.global_bias)


def compute_batch_implicit(model, u_batch, user_items_dict):
    """Compute implicit adjustment per instance in batch."""
    unique_users, inverse = torch.unique(u_batch, return_inverse=True)
    unique_np = unique_users.cpu().numpy()
    implicit_per_user = model.implicit_sum(unique_np, user_items_dict)  # (n_unique, dim)
    return implicit_per_user[inverse]                                   # (B, dim)


def build_user_items_dict(train_df):
    """user_idx -> list of items (from training history)."""
    return train_df.groupby("user_idx")["movie_idx"].apply(list).to_dict()


# ============================================================
# 5. TRAINING
# ============================================================
def train_bias_mf(model, train_df, val_df=None, epochs=20, batch_size=4096,
                  lr=1e-3, weight_decay=1e-5, patience=3, verbose=True):
    model.to(DEVICE)
    opt = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=weight_decay)

    u = torch.tensor(train_df["user_idx"].values,  dtype=torch.long)
    i = torch.tensor(train_df["movie_idx"].values, dtype=torch.long)
    r = torch.tensor(train_df[Config.RATING_COL].values, dtype=torch.float32)
    n = len(u)

    best_rmse, pc = float("inf"), 0
    for epoch in range(1, epochs+1):
        model.train()
        perm = torch.randperm(n)
        tot = 0.0
        for s in range(0, n, batch_size):
            idx = perm[s:s+batch_size]
            pred = model(u[idx].to(DEVICE), i[idx].to(DEVICE))
            loss = F.mse_loss(pred, r[idx].to(DEVICE))
            opt.zero_grad(); loss.backward(); opt.step()
            tot += loss.item() * len(idx)
        train_mse = tot / n

        val_rmse = _eval_bias_mf(model, val_df, batch_size) if val_df is not None else None
        if verbose:
            msg = f"  Epoch {epoch:2d} | TrainMSE={train_mse:.4f}"
            if val_rmse is not None: msg += f" | ValRMSE={val_rmse:.4f}"
            print(msg)

        if val_rmse is not None:
            if val_rmse < best_rmse - 1e-4:
                best_rmse = val_rmse; pc = 0
            else:
                pc += 1
                if pc >= patience:
                    if verbose: print(f"  Early stop @ epoch {epoch}")
                    break
    return model


def _eval_bias_mf(model, df, batch_size=4096):
    model.eval()
    with torch.no_grad():
        u = torch.tensor(df["user_idx"].values,  dtype=torch.long, device=DEVICE)
        i = torch.tensor(df["movie_idx"].values, dtype=torch.long, device=DEVICE)
        r = torch.tensor(df[Config.RATING_COL].values, dtype=torch.float32, device=DEVICE)
        mse = 0.0
        for s in range(0, len(u), batch_size):
            pred = model(u[s:s+batch_size], i[s:s+batch_size])
            mse += F.mse_loss(pred, r[s:s+batch_size], reduction="sum").item()
    return math.sqrt(mse / len(u))


def train_svdpp(model, train_df, val_df=None, user_items_dict=None,
                epochs=20, batch_size=4096, lr=1e-3, weight_decay=1e-5,
                patience=3, verbose=True):
    model.to(DEVICE)
    opt = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=weight_decay)

    u = torch.tensor(train_df["user_idx"].values,  dtype=torch.long)
    i = torch.tensor(train_df["movie_idx"].values, dtype=torch.long)
    r = torch.tensor(train_df[Config.RATING_COL].values, dtype=torch.float32)
    n = len(u)

    best_rmse, pc = float("inf"), 0
    for epoch in range(1, epochs+1):
        model.train()
        perm = torch.randperm(n)
        tot = 0.0
        for s in range(0, n, batch_size):
            idx = perm[s:s+batch_size]
            u_b = u[idx].to(DEVICE)
            i_b = i[idx].to(DEVICE)
            r_b = r[idx].to(DEVICE)

            implicit_batch = compute_batch_implicit(model, u_b, user_items_dict)
            pred = model(u_b, i_b, implicit_batch)
            loss = F.mse_loss(pred, r_b)
            opt.zero_grad(); loss.backward(); opt.step()
            tot += loss.item() * len(idx)
        train_mse = tot / n

        val_rmse = _eval_svdpp(model, val_df, user_items_dict, batch_size) if val_df is not None else None
        if verbose:
            msg = f"  Epoch {epoch:2d} | TrainMSE={train_mse:.4f}"
            if val_rmse is not None: msg += f" | ValRMSE={val_rmse:.4f}"
            print(msg)

        if val_rmse is not None:
            if val_rmse < best_rmse - 1e-4:
                best_rmse = val_rmse; pc = 0
            else:
                pc += 1
                if pc >= patience:
                    if verbose: print(f"  Early stop @ epoch {epoch}")
                    break
    return model


def _eval_svdpp(model, df, user_items_dict, batch_size=4096):
    model.eval()
    with torch.no_grad():
        u = torch.tensor(df["user_idx"].values,  dtype=torch.long, device=DEVICE)
        i = torch.tensor(df["movie_idx"].values, dtype=torch.long, device=DEVICE)
        r = torch.tensor(df[Config.RATING_COL].values, dtype=torch.float32, device=DEVICE)
        mse = 0.0
        for s in range(0, len(u), batch_size):
            u_b = u[s:s+batch_size]
            i_b = i[s:s+batch_size]
            r_b = r[s:s+batch_size]
            implicit_batch = compute_batch_implicit(model, u_b, user_items_dict)
            pred = model(u_b, i_b, implicit_batch)
            mse += F.mse_loss(pred, r_b, reduction="sum").item()
    return math.sqrt(mse / len(u))


# ============================================================
# 6. TEST EVAL — RMSE / MAE / R²
# ============================================================
def evaluate_rating(model, test_df, user_items_dict=None, is_svdpp=False, batch_size=4096):
    model.eval()
    preds, actuals = [], []
    with torch.no_grad():
        u = torch.tensor(test_df["user_idx"].values,  dtype=torch.long, device=DEVICE)
        i = torch.tensor(test_df["movie_idx"].values, dtype=torch.long, device=DEVICE)
        r = torch.tensor(test_df[Config.RATING_COL].values, dtype=torch.float32, device=DEVICE)

        for s in range(0, len(u), batch_size):
            u_b = u[s:s+batch_size]
            i_b = i[s:s+batch_size]
            if is_svdpp:
                implicit_batch = compute_batch_implicit(model, u_b, user_items_dict)
                pred = model(u_b, i_b, implicit_batch)
            else:
                pred = model(u_b, i_b)
            preds.extend(pred.cpu().numpy().flatten())
            actuals.extend(r[s:s+batch_size].cpu().numpy().flatten())

    preds, actuals = np.array(preds), np.array(actuals)
    return {
        "rmse": float(np.sqrt(mean_squared_error(actuals, preds))),
        "mae":  float(mean_absolute_error(actuals, preds)),
        "r2":   float(r2_score(actuals, preds)),
        "n_test": int(len(actuals)),
    }


# ============================================================
# 7. MAIN
# ============================================================
def main():
    train_df, val_df, test_df, user_map, movie_map = preprocess(
        Config.RATINGS_PATH,
        max_rows=Config.MAX_ROWS_TO_PROCESS,
        min_interactions=Config.MIN_INTERACTIONS,
    )
    num_users, num_movies = len(user_map), len(movie_map)
    user_items_dict = build_user_items_dict(train_df)

    results = []

    # ---------- Bias MF ----------
    set_seed()
    print("\n" + "="*70 + "\n[Bias MF] Training...\n" + "="*70)
    m_bias = BiasMF(num_users, num_movies, dim=Config.EMBEDDING_DIM)

    t0 = time.time()
    m_bias = train_bias_mf(
        m_bias, train_df, val_df,
        epochs=Config.EPOCHS, batch_size=Config.BATCH_SIZE,
        lr=Config.LEARNING_RATE, weight_decay=Config.WEIGHT_DECAY,
        patience=Config.PATIENCE,
    )
    train_time_bias = time.time() - t0

    t0 = time.time()
    metrics_bias = evaluate_rating(m_bias, test_df)
    eval_time_bias = time.time() - t0

    print(f"  [Test] RMSE={metrics_bias['rmse']:.4f} | MAE={metrics_bias['mae']:.4f} | "
          f"R²={metrics_bias['r2']:.4f} | EvalTime={eval_time_bias:.2f}s")
    results.append({"model": "Bias MF", "train_time_sec": round(train_time_bias, 2),
                    "eval_time_sec": round(eval_time_bias, 2), **metrics_bias})

    # ---------- SVD++ ----------
    set_seed()
    print("\n" + "="*70 + "\n[SVD++] Training...\n" + "="*70)
    m_svdpp = SVDpp(num_users, num_movies, dim=Config.EMBEDDING_DIM)

    t0 = time.time()
    m_svdpp = train_svdpp(
        m_svdpp, train_df, val_df, user_items_dict,
        epochs=Config.EPOCHS, batch_size=Config.BATCH_SIZE,
        lr=Config.LEARNING_RATE, weight_decay=Config.WEIGHT_DECAY,
        patience=Config.PATIENCE,
    )
    train_time_svdpp = time.time() - t0

    t0 = time.time()
    metrics_svdpp = evaluate_rating(m_svdpp, test_df, user_items_dict, is_svdpp=True)
    eval_time_svdpp = time.time() - t0

    print(f"  [Test] RMSE={metrics_svdpp['rmse']:.4f} | MAE={metrics_svdpp['mae']:.4f} | "
          f"R²={metrics_svdpp['r2']:.4f} | EvalTime={eval_time_svdpp:.2f}s")
    results.append({"model": "SVD++", "train_time_sec": round(train_time_svdpp, 2),
                    "eval_time_sec": round(eval_time_svdpp, 2), **metrics_svdpp})

    # ---------- Comparison ----------
    df = pd.DataFrame(results).set_index("model")
    df["train_ratio"] = (df["train_time_sec"] / df["train_time_sec"].iloc[0]).round(2)
    df["rmse_delta"]  = (df["rmse"] - df["rmse"].iloc[0]).round(4)

    print("\n" + "="*90)
    print("COMPARISON: Bias MF vs SVD++ (Rating Prediction)")
    print("="*90)
    cols = ["train_time_sec", "eval_time_sec", "train_ratio", "rmse", "mae", "r2", "rmse_delta"]
    print(df[cols].round(4).to_string())

    out = RUN_DIR / f"{Config.RUN_TIMESTAMP}_biasmf_vs_svdpp.csv"
    df.round(4).to_csv(out)
    print(f"\n[Saved] {out}")

    return df, m_bias, m_svdpp


if __name__ == "__main__":
    df_results, m_bias, m_svdpp = main()

[Config] Device=cpu | Run=20261023_151655_1000000
[Prep] Reading 1,000,000 newest...
[Prep] Train=671,078 | Val=13,140 | Test=13,140
[Prep] Users=13,140 | Items=2,557 | Rating range=[0.5, 5.0]

[Bias MF] Training...
  Epoch  1 | TrainMSE=10.5420 | ValRMSE=2.5038
  Epoch  2 | TrainMSE=2.2402 | ValRMSE=1.1606
  Epoch  3 | TrainMSE=0.9021 | ValRMSE=0.9648
  Epoch  4 | TrainMSE=0.7877 | ValRMSE=0.9229
  Epoch  5 | TrainMSE=0.7575 | ValRMSE=0.9075
  Epoch  6 | TrainMSE=0.7434 | ValRMSE=0.8993
  Epoch  7 | TrainMSE=0.7347 | ValRMSE=0.8957
  Epoch  8 | TrainMSE=0.7270 | ValRMSE=0.8917
  Epoch  9 | TrainMSE=0.7192 | ValRMSE=0.8880
  Epoch 10 | TrainMSE=0.7101 | ValRMSE=0.8823
  Epoch 11 | TrainMSE=0.6996 | ValRMSE=0.8788
  Epoch 12 | TrainMSE=0.6887 | ValRMSE=0.8749
  Epoch 13 | TrainMSE=0.6783 | ValRMSE=0.8712
  Epoch 14 | TrainMSE=0.6693 | ValRMSE=0.8682
  Epoch 15 | TrainMSE=0.6614 | ValRMSE=0.8665
  Epoch 16 | TrainMSE=0.6544 | ValRMSE=0.8642
  Epoch 17 | TrainMSE=0.6474 | ValRMSE=0.8611
 

In [3]:
from surprise import SVDpp, Dataset, Reader
import surprise

train_df, val_df, test_df, user_map, movie_map = preprocess(
    Config.RATINGS_PATH,
    max_rows=Config.MAX_ROWS_TO_PROCESS,
    min_interactions=Config.MIN_INTERACTIONS,
)
num_users, num_movies = len(user_map), len(movie_map)
user_items_dict = build_user_items_dict(train_df)


# PyTorch
# t0 = time.time()
# m_pytorch = train_svdpp_pytorch(train_df, val_df)
# t_pytorch = time.time() - t0

# Surprise
surprise_trainset = surprise.Dataset.load_from_df(
    train_df[[Config.USER_COL, Config.MOVIE_COL, Config.RATING_COL]],
    reader=surprise.Reader(rating_scale=(train_df[Config.RATING_COL].min(),
                                         train_df[Config.RATING_COL].max()))
).build_full_trainset()
t0 = time.time()
m_surprise = SVDpp(n_factors=64, n_epochs=30).fit(surprise_trainset)
t_surprise = time.time() - t0
print(f"Surprise SVD++ training time: {t_surprise:.2f}s")
# print(f"PyTorch: {t_pytorch:.1f}s | Surprise: {t_surprise:.1f}s | Speedup: {t_pytorch/t_surprise:.1f}x")

[Prep] Reading 1,000,000 newest...
[Prep] Train=671,078 | Val=13,140 | Test=13,140
[Prep] Users=13,140 | Items=2,557 | Rating range=[0.5, 5.0]
Surprise SVD++ training time: 483.64s
